In [2]:
import asdf
import dkist
import numpy as np
from astropy.modeling import models
from astropy import coordinates as coord
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS
from gwcs import wcs as gwcs
from gwcs import coordinate_frames as cf
from sunpy.coordinates import Helioprojective
from astropy.time import Time

# https://gwcs.readthedocs.io/en/latest/
# https://jwst-pipeline.readthedocs.io/en/latest/jwst/assign_wcs/asdf-howto.html
# https://bitbucket.org/dkistdc/dkist-inventory/src/master/dkist_inventory/transforms.py

# Create a simplified GWCS based on a single input FITS

In [3]:
ref = fits.open("/Users/seanhess/Data/level2/generated/pid_1_118/inv.SD9T3L/INV_SD9T3L_2022_06_02T22_14_11_260_L2.fits")
ref.info()
wcs = WCS(ref[1].header)
wcs

Filename: /Users/seanhess/Data/level2/generated/pid_1_118/inv.SD9T3L/INV_SD9T3L_2022_06_02T22_14_11_260_L2.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      76   ()      
  1  Log of Optical Depth at 500nm    1 ImageHDU        69   (81, 237, 1)   float32   
  2  Temperature    1 ImageHDU        69   (81, 237, 1)   float32   
  3  Electron Pressure    1 ImageHDU        69   (81, 237, 1)   float32   
  4  Microturbulence    1 ImageHDU        69   (81, 237, 1)   float32   
  5  Magnetic Field Strength    1 ImageHDU        69   (81, 237, 1)   float32   
  6  Line-of-sight Velocity    1 ImageHDU        69   (81, 237, 1)   float32   
  7  Magnetic Field Inclination (w.r.t. line-of-sight)    1 ImageHDU        69   (81, 237, 1)   float32   
  8  Magnetic Field Azimuth (w.r.t. line-of-sight)    1 ImageHDU        69   (81, 237, 1)   float32   
  9  Geometric Height above solar surface (tau ~ 1 at 500nm)    1 ImageHDU        69   (81, 237, 1) 

WCS Keywords

Number of WCS axes: 3
CTYPE : 'LOGTAU' 'HPLT-TAN' 'HPLN-TAN' 
CRVAL : np.float64(0.0) np.float64(-0.11305563055555555) np.float64(-0.13333571666666666) 
CRPIX : np.float64(12.0) np.float64(28.571428) np.float64(14.520161) 
PC1_1 PC1_2 PC1_3  : np.float64(1.0) np.float64(0.0) np.float64(0.0) 
PC2_1 PC2_2 PC2_3  : np.float64(0.0) np.float64(1.0) np.float64(0.0) 
PC3_1 PC3_2 PC3_3  : np.float64(0.0) np.float64(0.0) np.float64(1.0) 
CDELT : np.float64(-0.1) np.float64(0.000415055) np.float64(5.929356944444445e-05) 
NAXIS : 81  237  1

In [32]:
spatial_wcs = wcs.dropaxis(0)
spatial_wcs

WCS Keywords

Number of WCS axes: 2
CTYPE : 'HPLT-TAN' 'HPLN-TAN' 
CRVAL : np.float64(-0.11305563055555555) np.float64(-0.13333571666666666) 
CRPIX : np.float64(28.571428) np.float64(14.520161) 
PC1_1 PC1_2  : np.float64(1.0) np.float64(0.0) 
PC2_1 PC2_2  : np.float64(0.0) np.float64(1.0) 
CDELT : np.float64(0.000415055) np.float64(5.929356944444445e-05) 
NAXIS : 237  1

In [4]:
optical_wcs = wcs.sub(1)
optical_wcs

WCS Keywords

Number of WCS axes: 1
CTYPE : 'TAU--LOG' 
CRVAL : np.float64(0.0) 
CRPIX : np.float64(12.0) 
PC1_1  : np.float64(1.0) 
CDELT : np.float64(0.1) 
NAXIS : 81

In [1]:
# 0 = Optical Depth
# 1 = Spatial - Slit Position (Latitude)
# 2 = Spatial - Frame Y (longitude)
#print(wcs.wcs)



# Spatial Coordinates
# https://bitbucket.org/dkistdc/dkist-inventory/src/207ddeec1c331cf56acee833edc5ad355ea83e7d/dkist_inventory/transforms.py#lines-72
# TODO: units, using cunit
def spatial_transform(wcs):
    # MAKE SURE TO DROP THE AXES PROPERLY
    crpix = (wcs.wcs.crpix - 1)
    cdelt = wcs.wcs.cdelt
    crval = wcs.wcs.crval
    pc = wcs.wcs.pc
    
    lonpole = 180 # Read LONPOLE header
    ref_lat = crval[0]
    ref_lon = crval[1]

    print(crpix+1)
    print(crval)
    print(cdelt)

    # https://github.com/DKISTDC/dkist/blob/main/dkist/wcs/models.py#L93
    shift = models.Shift(-crpix[0]) & models.Shift(-crpix[1])
    scale = models.Scale(cdelt[0]) & models.Scale(cdelt[1])
    rotate = models.AffineTransformation2D(matrix=pc)
    projection = models.Pix2Sky_TAN()


    # print(ref_lon, ref_lat, lonpole)
    sky_rotate = models.RotateNative2Celestial(ref_lon, ref_lat, lonpole)


    return shift | scale | rotate | projection | sky_rotate



spatial_transform(spatial_wcs)

NameError: name 'spatial_wcs' is not defined

In [24]:
def optical_depth_transform(wcs):
    crpix = wcs.wcs.crpix[0] - 1
    cdelt = wcs.wcs.cdelt[0]
    crval = wcs.wcs.crval[0] # Not necessary, optical depth reference is 0
    return models.Shift(-crpix) | models.Scale(cdelt) # | models.Shift(crval)

optical_depth_transform(optical_wcs)

<CompoundModel(offset_0=-11., factor_1=0.1)>

In [25]:


# https://bitbucket.org/dkistdc/dkist-inventory/src/207ddeec1c331cf56acee833edc5ad355ea83e7d/dkist_inventory/transforms.py#lines-547
obstime = Time('2022-06-02T21:47:26.641000') # Time(self.parser.midpoint_header["DATE-AVG"])
obsgeo = [-5466045.5, -2404388.8, 2242134.0] # [self.parser.midpoint_header[k] for k in ("OBSGEO-X", "OBSGEO-Y", "OBSGEO-Z")]
observer = coord.ITRS(coord.CartesianRepresentation(*obsgeo * u.m), obstime=obstime)

# Is there a better way to define this frame?
optical_depth_frame = cf.CoordinateFrame(
    naxes=1, 
    axes_type=("optical_depth",),  # Optical depth doesn't exactly match SPATIAL, but no "better" option
    axes_order=[0],
    axes_names=("optical_depth",), 
    unit=(u.pix,),
    name="optical_depth"
)


# https://bitbucket.org/dkistdc/dkist-inventory/src/207ddeec1c331cf56acee833edc5ad355ea83e7d/dkist_inventory/transforms.py#lines-567
#celestial_frame = cf.CelestialFrame(
#    axes_order=(1,2),
#    name="helioprojective",
#    reference_frame=Helioprojective(obstime=obstime, observer=observer), 
#    axes_names=("helioprojective longitude", "helioprojective latitude"),
#    axis_physical_types=(
#        "custom:pos.helioprojective.lon",
#        "custom:pos.helioprojective.lat",
#    )
#)

pixel_frame = cf.CoordinateFrame(
    naxes=3,
    axes_type=["PIXEL"] * 3,
    axes_order=range(3),
    unit=[u.pix] * 3,
    axes_names=["optical_depth", "spatial along slit", "raster scan step number"],
    name="pixel",
)

# TODO: does this use ALL the DATE-AVG to make a reference for all of them, or just a single one?
# https://bitbucket.org/dkistdc/dkist-inventory/src/207ddeec1c331cf56acee833edc5ad355ea83e7d/dkist_inventory/transforms.py#lines-527
temporal_frame = cf.TemporalFrame(
    axes_order=[3],
    name="temporal",
    axes_names=("time",),
    unit=(u.s,),
    reference_frame=obstime,
)


helio = Helioprojective(
    obstime=obstime, 
    observer=observer,
)


sky_frame = cf.CelestialFrame(
    reference_frame=helio, 
    unit=(u.deg, u.deg),
    axes_order=(1,2),
    name="Helioprojective"
)

sky_frame



<CelestialFrame(name="Helioprojective", unit=(Unit("deg"), Unit("deg")), axes_names=('', ''), axes_order=(1, 2), reference_frame=<Helioprojective Frame (obstime=2022-06-02T21:47:26.641, rsun=695700.0 km, observer=<HeliographicStonyhurst Coordinate (obstime=2022-06-02T21:47:26.641, rsun=695700.0 km): (lon, lat, radius) in (deg, deg, m)
    (0.00035362, -0.46772335, 1.51723526e+11)>)>)>

In [30]:
composite_transform = optical_depth_transform(optical_wcs) & spatial_transform(spatial_wcs)
composite_transform

[28.571428 14.520178]
[-0.11305589 -0.13333601]
[4.15055000e-04 5.92935694e-05]


<CompoundModel(offset_0=-11., factor_1=0.1, offset_2=-27.571428, offset_3=-13.520178, factor_4=0.00000012, factor_5=0.00000002, matrix_6=[[1., 0.], [0., 1.]], translation_6=[0., 0.], lon_8=-0.13333601, lat_8=-0.11305589, lon_pole_8=180.)>

In [27]:
#pipeline1=gwcs.WCS([
#    (pixel_frame, composite_transform),
#    (cf.CompositeFrame([optical_depth_frame, celestial_frame]), None)
#])

quantity_gwcs=gwcs.WCS([
    (pixel_frame, composite_transform),
    (cf.CompositeFrame([optical_depth_frame, sky_frame]), None)
])

### Test out the pipeline

In [29]:
quantity_gwcs(0,1,2)

(-1.1, 359.8666609253818, -0.11305607863124086)

In [52]:
pipeline2.pixel_to_world(0,1,2)

[<Quantity -1.1 pix>,
 <SkyCoord (ICRS): (ra, dec) in deg
     (359.85563536, -0.11373896)>]

In [139]:
wcs2.pixel_to_world(0,1,2)

[<Quantity -1.1 optical_depth>,
 <SkyCoord (ICRS): (ra, dec) in deg
     (359.85563537, -0.11373896)>]

In [138]:
l2asdf = asdf.open("/Users/seanhess/Data/level2/generated/pid_1_118/inv.EGH11D/INV_EGH11D_L2.asdf")
wcs2 = l2asdf["inversion"]["quantities"]["wcs"]
wcs2(0,1,2)

(-1.1, 359.8556364744068, -0.113738800288907)

## Profiles GWCS

In [11]:
ref = fits.open("/Users/seanhess/Data/level2/generated/pid_1_118/inv.EGH11D/INV_EGH11D_2022_06_02T21_47_26_508_L2.fits")
ref.info()
wcsp = WCS(ref[12].header)
wcsp

Filename: /Users/seanhess/Data/level2/generated/pid_1_118/inv.EGH11D/INV_EGH11D_2022_06_02T21_47_26_508_L2.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      76   ()      
  1  Log of Optical Depth at 500nm    1 ImageHDU        69   (81, 237, 1)   float32   
  2  Temperature    1 ImageHDU        69   (81, 237, 1)   float32   
  3  Electron Pressure    1 ImageHDU        69   (81, 237, 1)   float32   
  4  Microturbulence    1 ImageHDU        69   (81, 237, 1)   float32   
  5  Magnetic Field Strength    1 ImageHDU        69   (81, 237, 1)   float32   
  6  Line-of-sight Velocity    1 ImageHDU        69   (81, 237, 1)   float32   
  7  Magnetic Field Inclination (w.r.t. line-of-sight)    1 ImageHDU        69   (81, 237, 1)   float32   
  8  Magnetic Field Azimuth (w.r.t. line-of-sight)    1 ImageHDU        69   (81, 237, 1)   float32   
  9  Geometric Height above solar surface (tau ~ 1 at 500nm)    1 ImageHDU        69   (81, 237, 1) 

WCS Keywords

Number of WCS axes: 4
CTYPE : 'STOKES' 'AWAV' 'HPLT-TAN' 'HPLN-TAN' 
CRVAL : np.float64(1.0) np.float64(6.301490000000001e-07) np.float64(-0.11305588888888889) np.float64(-0.1333360111111111) 
CRPIX : np.float64(1.0) np.float64(32.5625) np.float64(28.571428) np.float64(14.520178) 
PC1_1 PC1_2 PC1_3 PC1_4  : np.float64(1.0) np.float64(0.0) np.float64(0.0) np.float64(0.0) 
PC2_1 PC2_2 PC2_3 PC2_4  : np.float64(0.0) np.float64(1.0) np.float64(0.0) np.float64(0.0) 
PC3_1 PC3_2 PC3_3 PC3_4  : np.float64(0.0) np.float64(0.0) np.float64(1.0) np.float64(0.0) 
PC4_1 PC4_2 PC4_3 PC4_4  : np.float64(0.0) np.float64(0.0) np.float64(0.0) np.float64(1.0) 
CDELT : np.float64(1.0) np.float64(1.2800000000000002e-12) np.float64(0.000415055) np.float64(5.929356944444445e-05) 
NAXIS : 4  150  237  1

In [12]:
stokes_wcs = wcsp.sub(1)
spectral_wcs = wcsp.dropaxis(0).sub(1)
binned_wcs = wcsp.dropaxis(0).dropaxis(0)
binned_wcs

WCS Keywords

Number of WCS axes: 2
CTYPE : 'HPLT-TAN' 'HPLN-TAN' 
CRVAL : np.float64(-0.11305588888888889) np.float64(-0.1333360111111111) 
CRPIX : np.float64(28.571428) np.float64(14.520178) 
PC1_1 PC1_2  : np.float64(1.0) np.float64(0.0) 
PC2_1 PC2_2  : np.float64(0.0) np.float64(1.0) 
CDELT : np.float64(0.000415055) np.float64(5.929356944444445e-05) 
NAXIS : 237  1

In [13]:
def spectral_transform(wcs):
    crpix = wcs.wcs.crpix[0] - 1
    cdelt = wcs.wcs.cdelt[0]
    crval = wcs.wcs.crval[0]
    return models.Shift(-crpix) | models.Scale(cdelt) | models.Shift(crval)

spectral_transform(spectral_wcs)

<CompoundModel(offset_0=-31.5625, factor_1=0., offset_2=0.00000063)>

In [14]:
def stokes_transform(wcs):
    return models.Identity(1)

stokes_transform(stokes_wcs)

<Identity(1)>

In [15]:
binned_composite = stokes_transform(stokes_wcs) & spectral_transform(spectral_wcs) & spatial_transform(binned_wcs)
binned_composite

[28.571428 14.520178]
[-0.11305589 -0.13333601]
[4.15055000e-04 5.92935694e-05]


<CompoundModel(offset_1=-31.5625, factor_2=0., offset_3=0.00000063, offset_4=-27.571428, offset_5=-13.520178, factor_6=0.00041506, factor_7=0.00005929, matrix_8=[[1., 0.], [0., 1.]], translation_8=[0., 0.], lon_10=-0.13333601, lat_10=-0.11305589, lon_pole_10=180.)>

In [16]:
stokes_frame = cf.StokesFrame(
    axes_order=[0],
    axes_names=("stokes"), 
    name="stokes"
)

spectral_frame = cf.SpectralFrame(
    axes_order=[1],
    axes_names=("wavelength"), 
    name="wavelength"
)

binned_frame = cf.CoordinateFrame(
    naxes=4,
    axes_type=["PIXEL"] * 4,
    axes_order=range(4),
    unit=[u.pix] * 4,
    axes_names=["stokes", "wavelength", "spatial along slit", "raster scan step number"],
    name="pixel",
)

binned_sky_frame = cf.CelestialFrame(
    reference_frame=coord.ICRS(), 
    name='icrs', 
    unit=(u.deg, u.deg),
    axes_order=(2,3)
)


In [17]:
profile_gwcs=gwcs.WCS([
    (binned_frame, binned_composite),
    (cf.CompositeFrame([stokes_frame, spectral_frame, binned_sky_frame]), None)
])

In [18]:
profile_gwcs(1,0,1,2)

(1.0, 6.301086000000001e-07, 359.85563536324713, -0.11373895925605489)

## Write Example to File

In [1]:
out = asdf.AsdfFile()
out.tree['quantity'] = quantity_gwcs
out.tree['profile'] = profile_gwcs
out.write_to("./output/gwcs.asdf")

NameError: name 'asdf' is not defined

In [20]:
l2asdf = asdf.open("/Users/seanhess/Data/level2/generated/pid_1_118/inv.EGH11D/INV_EGH11D_L2.asdf")
wcs2_quantity = l2asdf["inversion"]["quantities"]["opticalDepth"]["wcs"]
wcs2_quantity(0,1,2)

#wcs2_profile = l2asdf["inversion"]["profiles"]["wcs"]
#wcs2_profile(1,0,1,2)

(-1.1, 359.8556364744068, -0.113738800288907)